# HOMEWORK 16 - Course Project

Done by: **Bohdan Pinchuk**

Link: https://github.com/BogdanPinchuk/ComputerVision-PBY_HW16_Course_Project

Stub

In [ ]:
# template

# Input data

# Solution

# Print results


In [ ]:
# save
import cv2
import apps.main as mn
from torch.utils.data import DataLoader
# train_dataset = mn.StreamPatchDataset(train_c_files, patch_size=patch_size)
# train_loader = DataLoader(train_dataset, batch_size=32, drop_last=False)
#
# def objective_gaussian(trial: optuna.Trial) -> float:
#     # Простір гіперпараметрів
#     ksize = trial.suggest_categorical('ksize', [3, 5, 7, 9])
#     sigma_x = trial.suggest_float('sigma_x', 0.1, 5.0)
#
#     model = GaussianFilter(ksize=ksize, sigma_x=sigma_x, device=device)
#
#     # Оцінка на валідаційному спліті (або побатчево з прунінгом)
#     val_psnr = evaluate_filter(model, val_loader, device=device)
#
#     # Звіт проміжного значення для Pruner (як на 1-му скріншоті)
#     trial.report(val_psnr, step=0)
#     if trial.should_prune():
#         raise optuna.TrialPruned()
#
#     return val_psnr

In [ ]:
# Spatial filtering baselines — Gaussian

import optuna
import numpy as np
import apps.reporter as rpt
import apps.spatial_filters as sf

# Input data
d_ksize = 1
step_ksize = 2
d_sigma_x = 0.1
step_sigma_x = 0.1
search_space = {
    "ksize": list(range(3, 11 + d_ksize, step_ksize)),
    "sigma_x": np.arange(0.2, 3.0 + d_sigma_x, step_sigma_x).round(1).tolist(),
}

# Solution
optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective_gaussian(trial: optuna.Trial) -> float:
    """
    Optuna objective function: search best Gaussian parameters on train_loader.

    :param trial: Optuna trial instance
    :return: validation Mean Squared Error (MSE) to minimize
    """
    # hyperparameter search space
    ksize = int(trial.suggest_categorical("ksize", search_space["ksize"]))
    sigma_x = float(trial.suggest_categorical("sigma_x", search_space["sigma_x"]))

    # initialization spatial filter model
    model = sf.GaussianFilter(ksize=ksize, sigma_x=sigma_x)

    # evaluation via MSE
    return sf.evaluate_model(model, loader=train_loader)


# auto-stops when grid is exhausted
sampler = optuna.samplers.GridSampler(search_space, seed=rng_seed)
study_gaussian = optuna.create_study(direction="minimize", sampler=sampler)
study_gaussian.optimize(objective_gaussian, show_progress_bar=True)

# validation of the optimal Gaussian filter on val_loader patches
best_params_gaussian = study_gaussian.best_params
best_gaussian = sf.GaussianFilter(**best_params_gaussian)
val_mse_gaussian = sf.evaluate_model(best_gaussian, loader=val_loader)

rp = rpt.Reporter()
rp.tolerance = 4
rp.add_item("Filter", "Gaussian")
rp.add_item("Optimal ksize", best_params_gaussian["ksize"])
rp.add_item("Optimal sigma_x", best_params_gaussian["sigma_x"])
rp.add_item("Best Train MSE", rp.format_value(study_gaussian.best_value))
rp.add_item("Validation MSE", rp.format_value(val_mse_gaussian))
rp.add_item("Total Trials", str(len(study_gaussian.trials)))

# Print results
rp.print_pd_report("Spatial Filtering Baseline — Gaussian Optimization")
